# Word-Sense Disambiguation
#### Semi-supervised NLP project based on a Yarowsky-style bootstrapping approach.

In [1]:
import os, re, math
from collections import defaultdict, Counter
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
nltk.download("punkt", quiet=True)

# attempt to download the AP88 corpus from the original course link
# The link is currently unreachable, so this block is commented 
# import urllib.request
# corpus_url = "http://www.cs.columbia.edu/~smaskey/NLP-Corpus/AP88"
# urllib.request.urlretrieve(corpus_url, "AP88_downloaded.txt")
# print("AP88 corpus downloaded successfully.")



# step 1: load and clean corpus
corpus_dir = "Corpus-spell-AP88" #prev dowloaded 

def load_corpus(folder):
    text = []
    for f in os.listdir(folder):
        if f.startswith("ap88"):
            path = os.path.join(folder, f)
            with open(path, "r", encoding="latin-1", errors="ignore") as file:
                text.append(file.read())
    return " ".join(text)

raw_text = load_corpus(corpus_dir)
print("corpus loaded locally, length:", len(raw_text))

# step 2: basic preprocessing
clean_text = re.sub(r"<[^>]*>", " ", raw_text)
clean_text = re.sub(r"\s+", " ", clean_text.lower())
sentences = sent_tokenize(clean_text)
print("sentences:", len(sentences))

# step 3: create synthetic ambiguous words
synthetic_words = {
    "carspeech": ("car", "speech"),
    "treeviolence": ("tree", "violence"),
    "chairpaper": ("chair", "paper"),
    "headword": ("head", "word"),
    "plantwar": ("plant", "war"),
    "dogbuilding": ("dog", "building"),
    "foodcomputer": ("food", "computer"),
    "riverclock": ("river", "clock"),
    "mountainstep": ("mountain", "step")
}


sense_map_num = {amb: {s1: 1, s2: 2} for amb, (s1, s2) in synthetic_words.items()}

# step 4: build synthetic corpus and gold truth
synthetic_corpus = []
gold_map = {amb: {} for amb in synthetic_words}
sentence_index = {}

for s in sentences:
    toks = word_tokenize(s)
    new_toks = list(toks)
    local_senses = {}
    for i, w in enumerate(toks):
        for amb, (s1, s2) in synthetic_words.items():
            if w == s1:
                new_toks[i] = amb
                local_senses[amb] = 1
            elif w == s2:
                new_toks[i] = amb
                local_senses[amb] = 2
    if local_senses:
        new_s = " ".join(new_toks)
        synthetic_corpus.append(new_s)
        sentence_index[new_s] = len(synthetic_corpus) - 1
        for amb, sid in local_senses.items():
            gold_map[amb][new_s] = sid

print("synthetic sentences with ambiguity:", len(synthetic_corpus))

# step 5: extract features from a local context window
def extract_features(sentence, target, window=2):
    toks = word_tokenize(sentence)
    feats = []
    for i, w in enumerate(toks):
        if w == target:
            for j in range(max(0, i - window), min(len(toks), i + window + 1)):
                if j != i:
                    feats.append(f"colloc:{j - i}:{toks[j]}")
    return feats

# step 6: compute log-likelihood ratio for features
def compute_log_likelihood(feat_counts, epsilon=0.1):
    # add small constant ε for smoothing (stability)
    total_s1 = sum(v["s1"] for v in feat_counts.values()) + epsilon
    total_s2 = sum(v["s2"] for v in feat_counts.values()) + epsilon
    rules = []
    for feat, counts in feat_counts.items():
        c1 = counts["s1"] + epsilon
        c2 = counts["s2"] + epsilon
        if (c1 + c2) < 5:  # ignore weak evidence
            continue
        p1 = c1 / total_s1
        p2 = c2 / total_s2
        if p1 > p2:
            score = math.log(p1 / p2)
            sense = 1
        else:
            score = math.log(p2 / p1)
            sense = 2
        rules.append((feat, sense, abs(score)))
    rules.sort(key=lambda x: x[2], reverse=True)
    return rules

# step 7: bootstrapping
def yarowsky_bootstrap(sentences, target, seed_sets, max_iter=5):
    # 1. Initialize with seed rules
    # 2. Extract collocational features  
    # 3. Compute log-likelihood ratios
    # 4. Apply decision list
    # 5. Expand labeled set
    # 6. Repeat until convergence
    s1, s2 = synthetic_words[target]
    labeled = []
    # initialize from seed collocations
    for s in sentences:     
        if target not in s:
            continue
        if any(w in s for w in seed_sets.get(s1, [])):
            labeled.append((s, 1))
        elif any(w in s for w in seed_sets.get(s2, [])):
            labeled.append((s, 2))

    # soft convergence tracking
    growth_curve = []

    for it in range(max_iter):
        feat_counts = defaultdict(lambda: {"s1": 0, "s2": 0})
        # step a: collect feature statistics
        for s, sid in labeled:
            for f in extract_features(s, target):
                if sid == 1:
                    feat_counts[f]["s1"] += 1
                else:
                    feat_counts[f]["s2"] += 1
        # step b: train new decision list
        rules = compute_log_likelihood(feat_counts)
        newly = 0
        labeled_texts = {x for x, _ in labeled}
        # step c: classify new examples based on strongest rule
        for s in sentences:
            if target in s and s not in labeled_texts:
                toks = word_tokenize(s)
                for feat, sid, sc in rules:
                    _, dist, word = feat.split(":", 2)
                    if word in toks:
                        labeled.append((s, sid))
                        labeled_texts.add(s)
                        newly += 1
                        break
        growth_curve.append(newly)
        print(f"iteration {it+1}: newly labeled {newly}")
        if newly < 5:  # stop when almost stable
            break
    return rules, labeled, growth_curve

#step 8: apply one sense per discourse rule
def one_sense_per_discourse(labeled, index_map, block_size=20):
    block_senses = defaultdict(Counter)
    for s, sid in labeled:
        idx = index_map.get(s, 0)
        doc_id = idx // block_size
        block_senses[doc_id][sid] += 1
    dominant = {d: c.most_common(1)[0][0] for d, c in block_senses.items()}
    adjusted = []
    for s, _ in labeled:
        idx = index_map.get(s, 0)
        doc_id = idx // block_size
        adjusted.append((s, dominant[doc_id]))
    return adjusted

# step 9: evaluate automatically using gold data (token-level + sentence-level)
def automatic_evaluation(labeled, target, gold):
    g = gold[target]

    #token-level evaluation 
    total = len(g)  #counts sentences, may include multiple tokens
    assigned = 0
    correct = 0
    for s, sid in labeled:
        if s in g:
            assigned += 1
            if g[s] == sid:
                correct += 1
    acc_token = correct / assigned if assigned > 0 else 0.0
    cov_token = assigned / total if total > 0 else 0.0

    #sentence-level evaluation (adjusted version) 
    unique_total = len(set(g.keys()))
    unique_assigned = len(set([s for s, _ in labeled if s in g]))
    correct_unique = sum(1 for s, sid in labeled if s in g and g[s] == sid)
    acc_sentence = correct_unique / unique_assigned if unique_assigned > 0 else 0.0
    cov_sentence = unique_assigned / unique_total if unique_total > 0 else 0.0

    return {
        "total": total,
        "assigned": assigned,
        "correct": correct,
        "acc_token": acc_token,
        "cov_token": cov_token,
        "acc_sentence": acc_sentence,
        "cov_sentence": cov_sentence,
    }

# step 10: seed words for each sense
seed_sets = {
    "car": ["road", "engine", "street"],
    "speech": ["audience", "words", "talk"],
    "tree": ["forest", "leaf", "wood"],
    "violence": ["crime", "riot", "attack"],
    "chair": ["table", "seat", "wood"],
    "paper": ["article", "journal", "newspaper"],
    "head": ["face", "body", "hair"],
    "word": ["text", "language", "dictionary"],
    "plant": ["factory", "industry", "production"],
    "war": ["battle", "army", "soldier"],
    "dog": ["animal", "pet", "bark"],
    "building": ["construction", "floor", "roof"],
    "food": ["meal", "eat", "kitchen"],
    "computer": ["mouse", "machine", "software"],
    "river": ["water", "bridge", "flood"],
    "clock": ["time", "hour", "watch"],
    "mountain": ["hill", "peak", "climb"],
    "step": ["walk", "move", "stair"]
}
# step 11: run experiment for all ambiguous targets
results = {}
for amb in synthetic_words.keys():
    print("\n===", amb, "===")
    rules, labeled, curve = yarowsky_bootstrap(synthetic_corpus, amb, seed_sets, max_iter=10)
    labeled = one_sense_per_discourse(labeled, sentence_index)

    evals = automatic_evaluation(labeled, amb, gold_map)
    results[amb] = {
        "rules": rules[:10],
        "curve": curve,
        "evals": evals
    }

    print(f"total: {evals['total']} assigned: {evals['assigned']} correct: {evals['correct']}")
    print(f"token-level  -> accuracy: {evals['acc_token']:.4f} coverage: {evals['cov_token']:.4f}")
    print(f"sentence-lvl -> accuracy: {evals['acc_sentence']:.4f} coverage: {evals['cov_sentence']:.4f}")
    print("growth curve:", curve)

# step 12: final summary
print("\n--- SUMMARY ---")
for amb, res in results.items():
    s1, s2 = synthetic_words[amb]
    e = res["evals"]
    print(f"\n{amb} ({s1}/{s2})")
    print(f"  token-level:    accuracy {e['acc_token']:.4f} | coverage {e['cov_token']:.4f}")
    print(f"  sentence-level: accuracy {e['acc_sentence']:.4f} | coverage {e['cov_sentence']:.4f}")
    print("  top rules:")
    for feat, sid, score in res["rules"]:
        print("   ", feat, "-> sense", sid, "(score", f"{score:.2f})")


corpus loaded locally, length: 228081960
sentences: 1486784
synthetic sentences with ambiguity: 86576

=== carspeech ===
iteration 1: newly labeled 12933
iteration 2: newly labeled 0
total: 13992 assigned: 14029 correct: 8665
token-level  -> accuracy: 0.6176 coverage: 1.0026
sentence-lvl -> accuracy: 0.6193 coverage: 1.0000
growth curve: [12933, 0]

=== treeviolence ===
iteration 1: newly labeled 4759
iteration 2: newly labeled 0
total: 5164 assigned: 5177 correct: 3263
token-level  -> accuracy: 0.6303 coverage: 1.0025
sentence-lvl -> accuracy: 0.6319 coverage: 1.0000
growth curve: [4759, 0]

=== chairpaper ===
iteration 1: newly labeled 2955
iteration 2: newly labeled 0
total: 3275 assigned: 3292 correct: 2010
token-level  -> accuracy: 0.6106 coverage: 1.0052
sentence-lvl -> accuracy: 0.6137 coverage: 1.0000
growth curve: [2955, 0]

=== headword ===
iteration 1: newly labeled 10845
iteration 2: newly labeled 0
total: 11450 assigned: 11471 correct: 8001
token-level  -> accuracy: 0.6975